In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import GroupKFold

# Ensure mamba_model.py is in the same directory
from mamba_model import create_model

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    data_dir = '.'  
    save_dir = './checkpoints' 
    device = 'cpu' # Using CPU for evaluation
    
    # Model Architecture
    projected_space = 64
    d_state = 16
    dconv = 4
    e_fact = 2
    num_mambas = 1
    patch_len = 32
    dropout = 0.3
    tango = False
    reverse_flip = 1

args = Config()

# Colors: 0=Blue(Base), 1=Purple(Pre), 2=Red(Stress)
COLORS = ['#1f77b4', '#9467bd', '#d62728']
CMAP = ListedColormap(COLORS)

# ==========================================
# 2. EVALUATION PIPELINE
# ==========================================
def main():
    print("Loading datasets...")
    X = np.load(os.path.join(args.data_dir, 'WESAD_X_calibrated.npy'))
    y = np.load(os.path.join(args.data_dir, 'WESAD_y_labeled.npy'))
    sub = np.load(os.path.join(args.data_dir, 'WESAD_sub_labeled.npy'), allow_pickle=True)

    gkf = GroupKFold(n_splits=5)

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=sub)):
        current_fold_num = fold + 1
        ckpt_path = os.path.join(args.save_dir, f'fold_{current_fold_num}_best.pt')
        
        if not os.path.exists(ckpt_path):
            continue

        print(f"Generating Plots for Fold {current_fold_num}...")
        
        # Load Model
        model_config = {
            'enc_in': X.shape[1], 'seq_len': X.shape[2], 'num_class': 3,
            'projected_space': args.projected_space, 'd_state': args.d_state,
            'dconv': args.dconv, 'e_fact': args.e_fact, 'num_mambas': args.num_mambas,
            'dropout': args.dropout, 'patch_len': args.patch_len,
            'only_forward_scan': 0 if args.tango else 1, 'reverse_flip': args.reverse_flip, 'max_pooling': 0,
        }
        model, _ = create_model(model_config)
        
        # Clean prefix from compile() if necessary
        raw_state_dict = torch.load(ckpt_path, map_location=args.device, weights_only=True)
        clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in raw_state_dict.items()}
        model.load_state_dict(clean_state_dict)
        model = model.to(args.device)
        model.eval()

        # Isolate Test Subjects for this fold
        test_subs = np.unique(sub[test_idx])
        
        # Plot each subject in a separate figure
        for subject_id in test_subs:
            # Get indices for just this subject
            sub_mask = (sub == subject_id)
            X_sub = X[sub_mask]
            y_sub = y[sub_mask]
            
            # Predict
            X_tensor = torch.tensor(X_sub, dtype=torch.float32).to(args.device)
            with torch.no_grad():
                logits = model(X_tensor)
                preds = logits.argmax(dim=1).cpu().numpy()
            
            true_labels = y_sub.argmax(axis=1)
            
            # Since stride is 5s, each chunk index represents 5 seconds of time progression
            time_minutes = np.arange(len(preds)) * 5 / 60 

            # --- PLOTTING ---
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 4), sharex=True)
            
            # Ground Truth Ribbon
            ax1.imshow(true_labels[None, :], cmap=CMAP, vmin=0, vmax=2, aspect='auto', 
                       extent=[time_minutes[0], time_minutes[-1], 0, 1], interpolation='nearest')
            ax1.set_yticks([])
            ax1.set_title(f"Subject {subject_id} (Fold {current_fold_num}) - GROUND TRUTH", fontweight='bold', loc='left')
            
            # Prediction Ribbon
            ax2.imshow(preds[None, :], cmap=CMAP, vmin=0, vmax=2, aspect='auto', 
                       extent=[time_minutes[0], time_minutes[-1], 0, 1], interpolation='nearest')
            ax2.set_yticks([])
            ax2.set_title(f"Subject {subject_id} - MAMBA PREDICTION", fontweight='bold', loc='left')
            ax2.set_xlabel("Time (Minutes) [Chunk Stride Progression]", fontsize=12)

            # Add Legend
            legend_text = "Blue: Baseline | Purple: Pre-Stress | Red: Stress"
            fig.text(0.5, 0.01, legend_text, ha='center', va='bottom', fontsize=10, 
                     bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))

            plt.tight_layout(rect=[0, 0.05, 1, 1])
            plt.show()

if __name__ == '__main__':
    main()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupKFold

# Ensure mamba_model.py is in the same directory as your notebook
from mamba_model import create_model

# ==========================================
# 1. JUPYTER CONFIGURATION
# ==========================================
class Config:
    data_dir = './'  
    save_dir = './checkpoints' 
    
    batch_size = 128
    device = 'cpu' # Switching to CPU as requested
    
    # Model Architecture
    projected_space = 64
    d_state = 16
    dconv = 4
    e_fact = 2
    num_mambas = 1
    patch_len = 32
    dropout = 0.3
    tango = False
    reverse_flip = 1

args = Config()

if args.device == 'cuda':
    torch.set_float32_matmul_precision('high')

# ==========================================
# 2. DATASET & EVALUATION LOGIC
# ==========================================
class WESADTensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

@torch.no_grad()
def evaluate_fold(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y_batch.argmax(1).numpy())

    return np.array(all_labels), np.array(all_preds)

# ==========================================
# 3. MAIN EXECUTION CELL
# ==========================================
print(f"Executing on: {args.device.upper()}")
print("Loading datasets...")

X = np.load(os.path.join(args.data_dir, 'WESAD_X_calibrated.npy'))
y = np.load(os.path.join(args.data_dir, 'WESAD_y_labeled.npy'))
sub = np.load(os.path.join(args.data_dir, 'WESAD_sub_labeled.npy'), allow_pickle=True)

# Updated for the 3-Class System
class_names = ['Baseline', 'Pre-Stress', 'Stress']

gkf = GroupKFold(n_splits=5)
aggregated_metrics = {
    'accuracy': [], 'macro_f1': [], 'pre_stress_f1': [],
    'pre_stress_recall': [], 'pre_stress_precision': []
}

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=sub)):
    current_fold_num = fold + 1
    ckpt_path = os.path.join(args.save_dir, f'fold_{current_fold_num}_best.pt')
    
    if not os.path.exists(ckpt_path):
        print(f"\n[!] Checkpoint for Fold {current_fold_num} not found. Skipping...")
        continue

    test_subs = np.unique(sub[test_idx])
    print(f"\n{'='*70}")
    print(f"EVALUATING FOLD {current_fold_num} | Test Subjects: {test_subs.tolist()}")
    print(f"{'='*70}")

    X_test, y_test = X[test_idx], y[test_idx]
    
    test_loader = DataLoader(WESADTensorDataset(X_test, y_test), batch_size=args.batch_size, shuffle=False, num_workers=0)

    model_config = {
        'enc_in': X.shape[1], 'seq_len': X.shape[2], 'num_class': 3, # Updated to 3 classes
        'projected_space': args.projected_space, 'd_state': args.d_state,
        'dconv': args.dconv, 'e_fact': args.e_fact, 'num_mambas': args.num_mambas,
        'dropout': args.dropout, 'patch_len': args.patch_len,
        'only_forward_scan': 0 if args.tango else 1, 'reverse_flip': args.reverse_flip, 'max_pooling': 0,
    }
    
    model, _ = create_model(model_config)
    
    # -------------------------------------------------------------
    # Clean the torch.compile() prefix from the state_dict
    # -------------------------------------------------------------
    raw_state_dict = torch.load(ckpt_path, map_location=args.device, weights_only=True)
    clean_state_dict = {}
    for key, val in raw_state_dict.items():
        if key.startswith("_orig_mod."):
            clean_key = key.replace("_orig_mod.", "")
            clean_state_dict[clean_key] = val
        else:
            clean_state_dict[key] = val
            
    model.load_state_dict(clean_state_dict)
    # -------------------------------------------------------------
    
    model = model.to(args.device)

    true_labels, preds = evaluate_fold(model, test_loader, args.device)

    acc = accuracy_score(true_labels, preds)
    print(f"Overall Accuracy: {acc:.4f}\n")
    
    report_dict = classification_report(true_labels, preds, target_names=class_names, output_dict=True, zero_division=0)
    print(classification_report(true_labels, preds, target_names=class_names, digits=4, zero_division=0))

    # Updated constraint to 3 labels
    cm = confusion_matrix(true_labels, preds, labels=[0, 1, 2])
    
    pre_stress_total = np.sum(cm[1, :])
    if pre_stress_total > 0:
        correct_ps = cm[1, 1]
        conf_baseline = cm[1, 0]
        conf_stress = cm[1, 2]
        
        print(f"--- PRE-STRESS (CLASS 1) MISCLASSIFICATION ANALYSIS ---")
        print(f"Total True Pre-Stress Chunks: {pre_stress_total}")
        print(f"  -> Correctly Predicted: {correct_ps} ({(correct_ps/pre_stress_total)*100:.1f}%) [RECALL]")
        print(f"  -> Confused as Baseline: {conf_baseline} ({(conf_baseline/pre_stress_total)*100:.1f}%)")
        print(f"  -> Confused as Stress:   {conf_stress} ({(conf_stress/pre_stress_total)*100:.1f}%)")
    else:
        print("--- PRE-STRESS ANALYSIS ---\nNo Pre-Stress chunks in this fold.")

    aggregated_metrics['accuracy'].append(acc)
    aggregated_metrics['macro_f1'].append(report_dict['macro avg']['f1-score'])
    if 'Pre-Stress' in report_dict:
        aggregated_metrics['pre_stress_f1'].append(report_dict['Pre-Stress']['f1-score'])
        aggregated_metrics['pre_stress_recall'].append(report_dict['Pre-Stress']['recall'])
        aggregated_metrics['pre_stress_precision'].append(report_dict['Pre-Stress']['precision'])

print(f"\n{'='*70}")
print(f"FINAL AGGREGATED METRICS (Across {len(aggregated_metrics['accuracy'])} Folds)")
print(f"{'='*70}")
print(f"Avg Accuracy:       {np.mean(aggregated_metrics['accuracy']):.4f} ± {np.std(aggregated_metrics['accuracy']):.4f}")
print(f"Avg Macro F1:       {np.mean(aggregated_metrics['macro_f1']):.4f} ± {np.std(aggregated_metrics['macro_f1']):.4f}")
print(f"Avg Pre-Stress F1:  {np.mean(aggregated_metrics['pre_stress_f1']):.4f} ± {np.std(aggregated_metrics['pre_stress_f1']):.4f}")
print(f"Avg Pre-Stress Rec: {np.mean(aggregated_metrics['pre_stress_recall']):.4f} ± {np.std(aggregated_metrics['pre_stress_recall']):.4f}")
print(f"Avg Pre-Stress Pre: {np.mean(aggregated_metrics['pre_stress_precision']):.4f} ± {np.std(aggregated_metrics['pre_stress_precision']):.4f}")